In [1]:
import pandas as pd
import numpy as np

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [2]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)
data_gdynia.head()

,osm_id,addr_street,addr_number,ground_floor_group,ground_floor_function,upper_floor_group,upper_floor_function,number_of_storeys,is_mixed_use,build_year_est,arch_style,condition,max_height,is_heritage,source,confidence,date_of_checking,number_of_functions
id,,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,hotel,NaN,Hotel,7,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
2,890729731,Świętego Piotra,6D,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
3,890292040,Węglowa,22,NaN,centrum handlowe,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
4,890729732,Świętego Piotra,6,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
5,136885205,Tadeusza Wendy,15,NaN,centrum medyczne,NaN,Centrum Medyczne,6,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,2,2026-09-27,1


In [3]:
data_gdynia_long = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long.head()

,addr_street,ground_floor_function
id,,
1,Świętego Piotra,hotel
2,Świętego Piotra,mieszkaniowa wielorodzinna
3,Węglowa,centrum handlowe
4,Świętego Piotra,mieszkaniowa wielorodzinna
5,Tadeusza Wendy,centrum medyczne


2.

In [4]:
num_build_NoNan = data_gdynia.groupby("addr_street").agg(num_build=("osm_id", "size"))
num_build_Nan = data_gdynia.groupby("addr_street", dropna=False).agg(
    num_build=("osm_id", "size")
)

In [5]:
assert num_build_Nan["num_build"].sum() == len(data_gdynia)
assert (
    num_build_NoNan["num_build"].sum()
    == len(data_gdynia) - data_gdynia["addr_street"].isna().sum()
)

3.

In [6]:
data_gdynia_agg = data_gdynia.groupby("addr_street")["number_of_storeys"].agg(
    ["mean", "median", "min", "max"]
)
data_gdynia_agg.loc["Starowiejska", "mean"]

np.float64(4.47457627118644)

In [7]:
assert (
    data_gdynia_agg.loc["Starowiejska", "mean"]
    == data_gdynia.loc[
        data_gdynia["addr_street"] == "Starowiejska", "number_of_storeys"
    ].mean()
)

4.

In [8]:
data_gdynia_agg_4 = data_gdynia.groupby("addr_street").agg(
    num_build=("osm_id", "size"),
    avg_storeys=("number_of_storeys", "mean"),
    oldest_year=("build_year_est", "min"),
    num_styles=("arch_style", "nunique"),
)

In [9]:
assert (
    data_gdynia_agg_4["num_build"].sum()
    == len(data_gdynia) - data_gdynia["addr_street"].isna().sum()
)

5.

In [ ]:
multilocal_per_street = (
    data_gdynia.assign(
        is_multilocal=lambda d: (d["ground_floor_function"].str.count(",") + 1) > 1
    )
    .groupby("addr_street")
    .agg(
        num_build=("osm_id", "size"),
        num_multilocal=("is_multilocal", "sum"),
        share_multilocals=("is_multilocal", "mean"),
    )
)

In [ ]:
assert np.isclose(
    multilocal_per_street["share_multilocals"],
    multilocal_per_street["num_multilocal"] / multilocal_per_street["num_build"],
).all()

6.

In [12]:
styles_per_street = (
    data_gdynia.groupby(["addr_street", "arch_style"])
    .agg("size")
    .reset_index(name="size")
)

styles_per_street2 = data_gdynia.groupby(
    ["addr_street", "arch_style"], as_index=False
).agg("size")

In [13]:
assert styles_per_street.equals(styles_per_street2)

7.

In [14]:
dada_num_avg = data_gdynia.groupby(data_gdynia["build_year_est"] // 50 * 50).agg(
    num_build=("osm_id", "size"), avg_storeys=("number_of_storeys", "mean")
)

In [15]:
assert dada_num_avg["num_build"].sum() == len(data_gdynia)

In [16]:
dada_num_avg

,num_build,avg_storeys
build_year_est,,
1900,385,4.98961
1950,601,2.925125
2000,119,3.621849


8.

In [17]:
mask = (data_gdynia_agg_4["num_build"] > 30) & (data_gdynia_agg_4["avg_storeys"] > 3)
chosen_streets = data_gdynia_agg_4[mask]

data_gdynia_chosen_street = data_gdynia[
    data_gdynia["addr_street"].isin(chosen_streets.index)
]

In [18]:
assert chosen_streets["num_build"].sum() == len(data_gdynia_chosen_street)

9.

In [ ]:
facilities_group = (
    data_gdynia_long.reset_index()
    .groupby("ground_floor_function")
    .agg(
        total_facilities=("addr_street", "size"),
        total_buildings=("id", "nunique"),
        total_diff_streets=("addr_street", "nunique"),
    )
)


dad1 = facilities_group["total_facilities"].rename("count").sort_index()

dad2 = data_gdynia_long["ground_floor_function"].value_counts().sort_index()

In [20]:
assert dad1.equals(dad2)
assert facilities_group["total_facilities"].sum() == len(data_gdynia_long)

10.

In [ ]:
data_gdynia_avg_storeys = data_gdynia.groupby("addr_street").agg(
    average_from_buildings=("number_of_storeys", "mean")
)

data_gdynia_storeys = data_gdynia.loc[:, ["number_of_storeys"]]
data_gdynia_long_storeys = pd.merge(
    data_gdynia_long,
    data_gdynia_storeys,
    left_index=True,
    right_index=True,
    how="inner",
)
data_gdynia_long_avg_storeys = data_gdynia_long_storeys.groupby("addr_street").agg(
    average_from_premises=("number_of_storeys", "mean")
)

avg_build_per_street = pd.merge(
    data_gdynia_avg_storeys,
    data_gdynia_long_avg_storeys,
    left_index=True,
    right_index=True,
    how="inner",
)
avg_build_per_street = avg_build_per_street.assign(
    diff=lambda d: d["average_from_buildings"] - d["average_from_premises"]
)

avg_build_per_street.sort_values("diff", ascending=False)[:1]

,average_from_buildings,average_from_premises,diff
addr_street,,,
Świętego Wojciecha,7.0,6.333333,0.666667


In [ ]:
index_no_multilocal = multilocal_per_street[
    multilocal_per_street["num_multilocal"] == 0
].index
list_of_streets_no_multilocal = avg_build_per_street[
    avg_build_per_street.index.isin(index_no_multilocal)
]

In [ ]:
assert (list_of_streets_no_multilocal["diff"] == 0).sum() == len(
    list_of_streets_no_multilocal
)